This notebook loads saved PNG outputs and lets you switch between them in the same output area using clickable labels. If the results folder is not found automatically, edit the `results_dir` line in the next cell.

In [ ]:
from pathlib import Path
from IPython.display import display, Image, clear_output
import ipywidgets as widgets


def find_results_dir():
    """Find the Q75 results folder near the current notebook location."""
    here = Path.cwd().resolve()
    candidates = []

    for base in [here, *here.parents]:
        candidates.extend([
            base / "outputs" / "Q75",
            base / "dp_arve" / "outputs" / "Q75",
            base / "Q75",
        ])

    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "Could not find an outputs/Q75 folder automatically. "
        "Set results_dir manually in this cell."
    )


def categorize_png(path):
    name = path.name.lower()
    parts = [part.lower() for part in path.parts]

    if any(part == "confusion matrix" for part in parts) or name.startswith("cm_"):
        return "Confusion matrices"
    if name.startswith("scatter_"):
        return "Scatters"
    if name.startswith("timeseries_"):
        return "Timeseries"
    if name.startswith("flood_"):
        return "Flood events"
    return "Other"


def pretty_label(path, results_dir):
    rel = path.relative_to(results_dir).as_posix()
    return rel.replace(".png", "")


results_dir = find_results_dir()
png_files = sorted(results_dir.rglob("*.png"))

if not png_files:
    raise FileNotFoundError(f"No PNG files were found under {results_dir}")

files_by_category = {}
for png_path in png_files:
    category = categorize_png(png_path)
    files_by_category.setdefault(category, []).append(png_path)

category_order = [
    category
    for category in ["Timeseries", "Scatters", "Flood events", "Confusion matrices", "Other"]
    if category in files_by_category
]

category_buttons = widgets.ToggleButtons(
    options=category_order,
    description="Group:",
    button_style="",
    style={"description_width": "initial"},
)
file_buttons = widgets.ToggleButtons(
    options=[],
    description="Plot:",
    button_style="",
    style={"description_width": "initial"},
    layout=widgets.Layout(flex_flow="row wrap", width="100%"),
)
viewer = widgets.Output()
info = widgets.HTML()


def show_image(path_str):
    with viewer:
        clear_output(wait=True)
        display(Image(filename=path_str))


def refresh_files(*_):
    category = category_buttons.value
    category_files = files_by_category.get(category, [])
    options = {pretty_label(path, results_dir): str(path) for path in category_files}
    file_buttons.options = options
    if options:
        file_buttons.value = next(iter(options.values()))
    else:
        file_buttons.value = None
        with viewer:
            clear_output(wait=True)
    info.value = f"<b>Results folder:</b> {results_dir}<br><b>PNG count:</b> {len(png_files)}"


def on_file_change(change):
    if change["name"] == "value" and change["new"]:
        show_image(change["new"])


category_buttons.observe(refresh_files, names="value")
file_buttons.observe(on_file_change, names="value")

display(info)
display(category_buttons)
display(file_buttons)
display(viewer)

refresh_files()


HTML(value='')

ToggleButtons(description='Group:', options=('Timeseries', 'Scatters', 'Flood events', 'Confusion matrices'), …

ToggleButtons(description='Plot:', layout=Layout(flex_flow='row wrap', width='100%'), options=(), style=Toggle…

Output()